In [2]:
import cv2
import os
import time
import numpy as np
import pandas as pd
from scipy.ndimage import zoom
from tqdm import tqdm

window_width = 600
window_height = 1200

def reset_click_event():
    global points, holes, temp_img, img
    points.clear()
    holes.clear()
    temp_img = img.copy()
    cv2.imshow('image', temp_img)

def click_event(event, x, y, flags, params):
    global current_list, creating_hole
    if event == cv2.EVENT_LBUTTONDOWN:
        cv2.circle(temp_img, (x, y), 3, (0, 0, 255), -1)
        current_list.append((x, y))
        if len(current_list) >= 2:
            cv2.line(temp_img, current_list[-2], current_list[-1], (255, 0, 0), 2)
        cv2.imshow('image', temp_img)

def key_event(key):
    global current_list, creating_hole
    if key == ord('c'):  # Switch between main region and hole creation
        if creating_hole:
            creating_hole = False
            current_list = points
        else:
            creating_hole = True
            current_hole = []
            holes.append(current_hole)
            current_list = current_hole
    if key == ord('r'):
        reset_click_event()

Do_Mask = True

image_dir_path = r'/Users/coreyzhang/Documents/DISC_Files/Output/Cropped'
if Do_Mask:
    mask_file_path = os.path.join(image_dir_path, 'mask_x.npy')
else:
    mask_file_path = os.path.join(image_dir_path, 'mask_x.npy')

# Paths
csv_dir_path = os.path.join(image_dir_path, 'pydic/result')
modified_csv_dir_path = os.path.join(image_dir_path, 'csv')

if Do_Mask:
    # Global variables
    points = []       # For storing main region points
    holes = []        # List to store holes, each hole is a list of points
    current_list = points  # Reference to the current list being used (points or hole)
    creating_hole = False  # Flag to indicate if we are creating a hole
    img = None        # Original image
    temp_img = None   # Temporary image for drawing

    image_files = sorted([f for f in os.listdir(image_dir_path) if f.endswith('.png')])
    Imgs_for_mask = os.path.join(image_dir_path, image_files[0])

    img = cv2.imread(Imgs_for_mask)
    temp_img = img.copy()  # Create a temporary copy for drawing

    cv2.namedWindow('image', cv2.WINDOW_NORMAL)
    cv2.imshow('image', temp_img)

    # Set mouse callback function
    cv2.setMouseCallback('image', click_event)

    while True:
        key = cv2.waitKey(1) & 0xFF

        if key == 27:  # ESC key to exit
            break
        elif key == 13:  # Enter key to finish
            break
        key_event(key)

    cv2.destroyAllWindows()

    # Create a mask and draw the main polygon
    mask = np.zeros(img.shape[:2], dtype=np.uint8)  # Single channel for grayscale
    roi_corners = np.array([points], dtype=np.int32)
    cv2.fillPoly(mask, roi_corners, 255)  # Fill main region with white

    # Create holes in the mask
    for hole in holes:
        hole_corners = np.array([hole], dtype=np.int32)
        cv2.fillPoly(mask, hole_corners, 0)  # Fill hole with black

    # Convert mask to a Boolean array
    mask_bool = mask.astype(bool)

    # Invert the Boolean mask
    mask_bool_inverted = ~mask_bool

    # Save the inverted Boolean mask
    np.save(mask_file_path, mask_bool_inverted)

    # Convert the single channel mask to a 3-channel mask to match the original image
    mask_3channel = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

    # Apply the mask to the original image
    masked_image = cv2.bitwise_and(img, mask_3channel)

    # Save or display the result
    cv2.namedWindow('carved out region', cv2.WINDOW_NORMAL)
    cv2.imshow('carved out region', masked_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Ensure output directories exist
os.makedirs(modified_csv_dir_path, exist_ok=True)

# Load and downsize the mask
mask = np.load(mask_file_path)
mask = ~mask  # Invert mask

# Get list of image and csv files
csv_files = sorted([f for f in os.listdir(csv_dir_path) if f.endswith('.csv')])
image_files = sorted([f for f in os.listdir(image_dir_path) if f.endswith('.png')])

# Process each pair of files
for csv_file, image_file in zip(tqdm(csv_files, desc="Processing CSV files"), image_files):
    # Paths
    csv_file_path = os.path.join(csv_dir_path, csv_file)
    image_path = os.path.join(image_dir_path, image_file)
    modified_csv_path = os.path.join(modified_csv_dir_path, f'modified_{csv_file}')

    # Process as in your original code
    df = pd.read_csv(csv_file_path)
    x = df.iloc[:, 0].unique()
    y = df.iloc[:, 1].unique()
    resized_mask = zoom(mask, (len(y) / mask.shape[0], len(x) / mask.shape[1]), order=0)
    Z = df.pivot(index=df.columns[1], columns=df.columns[0], values=df.columns[-1])
    Z_masked = Z.where(resized_mask)
    df_masked = Z_masked.reset_index().melt(id_vars=[df.columns[1]], value_name=df.columns[-1])
    df_final = df.iloc[:, :-1].merge(df_masked, on=[df.columns[0], df.columns[1]])
    df_final.to_csv(modified_csv_path, index=False)
    
print('Oh Yeah!')


Processing CSV files: 100%|██████████| 1/1 [00:00<00:00,  7.56it/s]

Oh Yeah!
